In [ ]:
from pathlib import Path
import os
_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "cifar10"]
_dataset_root = next(p for p in _candidates if (p / "baseline_cifar10.py").exists())
os.chdir(_dataset_root)


In [ ]:
# Edge AI Energy Efficiency Project - PyTorch Analysis Notebook
# importing the required libraries
import os
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# importing PyTorch libraries
import torch
import torch.nn as nn
import torch.optim as optim

# torchvision is used here to load the CIFAR-10 dataset
from torchvision import datasets, transforms
# dataLoader helps us load the data in batches
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
# create project folders
folders = ["models", "results", "plots_analysis"]
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created or already exist:", folders)

In [ ]:
# if GPU is not available, use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# loading CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,) * 3, (0.3081,) * 3)
])
train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)
# batch_size means how many images the model sees at one time
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=1000,
    shuffle=False
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

In [ ]:
# PyTorch model
# the input image size is 32 x 32 x 3, so the total input features are 3072
class CIFAR10Model(nn.Module):
    def __init__(self):
        super(CIFAR10Model, self).__init__()

        # this changes the 32x32x3 image into one long vector
        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(32 * 32 * 3, 128)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.flatten(x)

        x = self.fc1(x)
        x = self.relu1(x)

        x = self.fc2(x)
        x = self.relu2(x)

        x = self.fc3(x)

        return x

# creating the model and sending it to CPU
model = CIFAR10Model().to(device)

print(model)

In [ ]:
# train the model
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# number of times the model goes through the training dataset
epochs = 5

train_losses = []
train_accuracies = []

start_time = time.time()

for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # calculating average loss and accuracy for this epoch
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(f"Epoch [{epoch + 1}/{epochs}] - Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%")

training_time = time.time() - start_time

print("Training completed")
print("Training time:", round(training_time, 2), "seconds")

In [ ]:
# graph for training loss
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), train_losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss Over Epochs")
plt.grid(True)
plt.savefig("plots_analysis/training_loss.png")
plt.show()

In [ ]:
# graph for training accuracy
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), train_accuracies, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Accuracy (%)")
plt.title("Training Accuracy Over Epochs")
plt.grid(True)
plt.savefig("plots_analysis/training_accuracy.png")
plt.show()

In [ ]:
# model evaluation and quantization
def evaluate_model(model, data_loader, device):
    model.eval()

    correct = 0
    total = 0

    start_time = time.time()

    # no_grad is used because we are only testing the model
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    inference_time = time.time() - start_time
    accuracy = 100 * correct / total

    return accuracy, inference_time

In [ ]:
# evaluating the original PyTorch model on the test dataset
original_model_path = "models/cifar10_pytorch_model.pt"
torch.save(model.state_dict(), original_model_path)

# checking original model size in KB
original_model_size = os.path.getsize(original_model_path) / 1024

print("Original model saved at:", original_model_path)
print("Original model size:", round(original_model_size, 2), "KB")

In [ ]:
# applying dynamic quantization which reduces the model size
model_cpu = copy.deepcopy(model).to("cpu")
quantized_model = torch.quantization.quantize_dynamic(
    model_cpu,
    {nn.Linear},
    dtype=torch.qint8
)

print("Quantized model created")
print(quantized_model)

In [ ]:
# saving the quantized model
quantized_model_path = "models/quantized_model_analysis.pt"
torch.save(quantized_model.state_dict(), quantized_model_path)

# checking quantized model size in KB
quantized_model_size = os.path.getsize(quantized_model_path) / 1024

print("Quantized model saved at:", quantized_model_path)
print("Quantized model size:", round(quantized_model_size, 2), "KB")

In [ ]:
# quantized model is evaluated on CPU
quantized_accuracy, quantized_inference_time = evaluate_model(
    quantized_model,
    test_loader,
    torch.device("cpu")
)

print("Quantized Model Accuracy:", round(quantized_accuracy, 2), "%")
print("Quantized Model Inference Time:", round(quantized_inference_time, 4), "seconds")

In [ ]:
# evaluating the original PyTorch model
original_accuracy, original_inference_time = evaluate_model(
    model,
    test_loader,
    device
)

print("Original Model Accuracy:", round(original_accuracy, 2), "%")
print("Original Model Inference Time:", round(original_inference_time, 4), "seconds")

In [ ]:
# comparing the original and quantized models
comparison_data = {
    "Model": ["Original PyTorch Model", "Quantized PyTorch Model"],
    "Accuracy (%)": [original_accuracy, quantized_accuracy],
    "Inference Time (seconds)": [original_inference_time, quantized_inference_time],
    "Model Size (KB)": [original_model_size, quantized_model_size]
}

comparison_df = pd.DataFrame(comparison_data)
comparison_df.to_csv("results/model_comparison.csv", index=False)
comparison_df

In [ ]:
# plotting model size comparison - smaller model size means the model is better for edge deployment
plt.figure(figsize=(7, 5))
plt.bar(comparison_df["Model"], comparison_df["Model Size (KB)"])
plt.ylabel("Model Size (KB)")
plt.title("Original vs Quantized Model Size")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("plots_analysis/model_size_comparison.png")
plt.show()

In [ ]:
# plotting accuracy comparison
plt.figure(figsize=(7, 5))
plt.bar(comparison_df["Model"], comparison_df["Accuracy (%)"])
plt.ylabel("Accuracy (%)")
plt.title("Original vs Quantized Model Accuracy")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("plots_analysis/accuracy_comparison.png")
plt.show()

In [ ]:
# plotting inference time comparison - comparing how fast the original and quantized models run
plt.figure(figsize=(7, 5))
plt.bar(comparison_df["Model"], comparison_df["Inference Time (seconds)"])
plt.ylabel("Inference Time (seconds)")
plt.title("Original vs Quantized Model Inference Time")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("plots_analysis/inference_time_comparison.png")
plt.show()

In [ ]:
# simulate multiple edge devices where each device gets a small part of the test dataset
num_devices = 5
samples_per_device = 1000

edge_results = []

for device_id in range(num_devices):
    start_index = device_id * samples_per_device
    end_index = start_index + samples_per_device

    # selecting a different part of the test dataset for each device
    device_subset = Subset(test_dataset, range(start_index, end_index))

    device_loader = DataLoader(
        device_subset,
        batch_size=100,
        shuffle=False
    )
    # testing the quantized model
    accuracy, inference_time = evaluate_model(
        quantized_model,
        device_loader,
        torch.device("cpu")
    )
    edge_results.append({
        "Edge Device": f"Device {device_id + 1}",
        "Samples Processed": samples_per_device,
        "Accuracy (%)": accuracy,
        "Inference Time (seconds)": inference_time
    })

edge_df = pd.DataFrame(edge_results)
edge_df.to_csv("results/edge_device_simulation.csv", index=False)
edge_df

In [ ]:
# plotting accuracy
plt.figure(figsize=(8, 5))
plt.bar(edge_df["Edge Device"], edge_df["Accuracy (%)"])
plt.xlabel("Edge Device")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Across Simulated Edge Devices")
plt.tight_layout()
plt.savefig("plots_analysis/edge_device_accuracy.png")
plt.show()

In [ ]:
# plotting inference time
plt.figure(figsize=(8, 5))
plt.bar(edge_df["Edge Device"], edge_df["Inference Time (seconds)"])
plt.xlabel("Edge Device")
plt.ylabel("Inference Time (seconds)")
plt.title("Inference Time Across Simulated Edge Devices")
plt.tight_layout()
plt.savefig("plots_analysis/edge_device_inference_time.png")
plt.show()

In [ ]:
# communication cost analysis
communication_data = {
    "Model": ["Original PyTorch Model", "Quantized PyTorch Model"],
    "Estimated Communication Cost (KB)": [
        original_model_size,
        quantized_model_size
    ]
}
communication_df = pd.DataFrame(communication_data)
communication_df.to_csv("results/communication_cost.csv", index=False)
communication_df

In [ ]:
# plotting communication cost comparison
plt.figure(figsize=(7, 5))
plt.bar(
    communication_df["Model"],
    communication_df["Estimated Communication Cost (KB)"]
)
plt.ylabel("Estimated Communication Cost (KB)")
plt.title("Estimated Communication Cost Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("plots_analysis/communication_cost_comparison.png")
plt.show()

In [ ]:
# federated learning communication estimate
device_counts = [1, 5, 10, 20]

federated_results = []

for count in device_counts:
    original_total_cost = original_model_size * count
    quantized_total_cost = quantized_model_size * count

    reduction = original_total_cost - quantized_total_cost
    reduction_percent = (reduction / original_total_cost) * 100

    federated_results.append({
        "Number of Edge Devices": count,
        "Original Model Communication Cost (KB)": original_total_cost,
        "Quantized Model Communication Cost (KB)": quantized_total_cost,
        "Reduction (%)": reduction_percent
    })

federated_df = pd.DataFrame(federated_results)
federated_df.to_csv("results/federated_communication_estimate.csv", index=False)
federated_df

In [ ]:
# plotting federated communication cost estimate
plt.figure(figsize=(8, 5))
plt.plot(
    federated_df["Number of Edge Devices"],
    federated_df["Original Model Communication Cost (KB)"],
    marker="o",
    label="Original Model"
)
plt.plot(
    federated_df["Number of Edge Devices"],
    federated_df["Quantized Model Communication Cost (KB)"],
    marker="o",
    label="Quantized Model"
)
plt.xlabel("Number of Edge Devices")
plt.ylabel("Communication Cost (KB)")
plt.title("Federated Learning Style Communication Cost Estimate")
plt.legend()
plt.grid(True)
plt.savefig("plots_analysis/federated_communication_cost.png")
plt.show()